# Track 3: Object Swaps and Loss

This notebook builds the CHAIR-compatible object co-occurrence matrix, generates structured and random-swap negatives, spot-checks 100 negatives, and smoke-tests the object-swap loss with pretrained ClipCap.

## Setup

Run the Track 1 setup first so Drive is mounted and the ClipCap model/tokenizer are available for the final loss cell.

In [7]:
from __future__ import annotations

import json
import os
import pickle
import random
import re
import subprocess
import sys
from collections.abc import Mapping, Sequence
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from google.colab import drive
import transformers
from transformers import GPT2Tokenizer

import nltk
import inflect

for pkg in ['punkt', 'punkt_tab', 'wordnet']:
    nltk.download(pkg, quiet=True)

drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/sc4001_nndl_grp_project'
TRACK1_ROOT = os.path.join(PROJECT_ROOT, 'phase_1', 'track_1')
TRACK3_ROOT = os.path.join(PROJECT_ROOT, 'phase_1', 'track_3')
os.makedirs(TRACK3_ROOT, exist_ok=True)

transformers.AdamW = torch.optim.AdamW
clipcap_dir = '/content/CLIP_prefix_caption'
if not os.path.isdir(clipcap_dir):
    subprocess.run(
        ['git', 'clone', '-q', 'https://github.com/rmokady/CLIP_prefix_caption', clipcap_dir],
        check=True,
    )
if clipcap_dir not in sys.path:
    sys.path.append(clipcap_dir)
from train import ClipCaptionPrefix, MappingType

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
def load_object_data(path: str | Path) -> dict:
    """Load and validate Track 1's coco_objects.json."""
    with open(path) as handle:
        data = json.load(handle)
    required = {'categories', 'synonyms', 'word_to_node', 'double_words'}
    missing = required.difference(data)
    if missing:
        raise ValueError(f'object data is missing keys: {sorted(missing)}')
    return data


def load_chair_gt(path: str | Path) -> dict:
    """Load Track 1's chair_gt_<split>.pkl cache."""
    with open(path, 'rb') as handle:
        return pickle.load(handle)


def true_objects(cache_entry: Mapping) -> set[str]:
    """Return segmentation objects unioned with every human-caption object."""
    objects = set(cache_entry.get('seg', []))
    for caption_objects in cache_entry.get('cap', []):
        objects.update(caption_objects)
    return objects


def build_cooccurrence_matrix(
    chair_gt: Mapping[int, Mapping],
    categories: Sequence[str],
) -> tuple[np.ndarray, dict[str, int]]:
    """Count image-level object co-occurrences, including the diagonal."""
    category_to_index = {category: i for i, category in enumerate(categories)}
    matrix = np.zeros((len(categories), len(categories)), dtype=np.int64)
    for entry in chair_gt.values():
        present = sorted(
            category_to_index[category]
            for category in true_objects(entry)
            if category in category_to_index
        )
        for row in present:
            matrix[row, present] += 1
    return matrix, category_to_index


def save_cooccurrence_matrix(
    path: str | Path,
    matrix: np.ndarray,
    categories: Sequence[str],
) -> None:
    """Save a portable matrix artifact with its category ordering."""
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    with open(target, 'wb') as handle:
        pickle.dump({'categories': list(categories), 'matrix': matrix}, handle)


# ---------------------------------------------------------------------------
# CHAIR-compatible object extraction (mirrors Track 1's caption_to_objects)
# ---------------------------------------------------------------------------
_INFLECT = inflect.engine()


def _singularize(word: str, object_words: set[str]) -> str:
    """Singularize, keeping vocabulary words as-is (inflect breaks bus -> bu)."""
    if word in object_words:
        return word
    singular = _INFLECT.singular_noun(word)
    return singular if singular else word


def caption_to_objects(caption: str, object_data: Mapping) -> tuple[list[str], list[str]]:
    """CHAIR-compatible object extraction (same logic as Track 1).

    Returns (object_words, node_words) where each entry is one mention in order.
    """
    word_to_node = object_data['word_to_node']
    double_words = object_data['double_words']
    object_words = set(word_to_node.keys())

    tokens = nltk.word_tokenize(caption.lower())
    words = [_singularize(t, object_words) for t in tokens]

    merged = []
    i = 0
    while i < len(words):
        if i + 1 < len(words):
            pair = words[i] + ' ' + words[i + 1]
            if pair in double_words:
                merged.append(double_words[pair])
                i += 2
                continue
        merged.append(words[i])
        i += 1

    if 'toilet' in merged and 'seat' in merged:
        merged = [w for w in merged if w != 'seat']

    obj_words, node_words = [], []
    for w in merged:
        if w in object_words:
            obj_words.append(w)
            node_words.append(word_to_node[w])
    return obj_words, node_words


# ---------------------------------------------------------------------------
# Span-based object finding for text replacement
# ---------------------------------------------------------------------------
def build_replacement_patterns(
    object_data: Mapping,
) -> list[tuple[re.Pattern, str]]:
    """Precompute regex patterns for finding object mentions and their spans.

    Sorted longest-first so multi-word phrases ("hot dog") match before their
    sub-words ("dog").  Each synonym word also gets a plural pattern so that
    "dogs" is found even though the vocabulary only lists "dog".
    """
    word_to_node = object_data['word_to_node']
    entries: dict[str, str] = {}
    for word, category in word_to_node.items():
        if word not in entries:
            entries[word] = category
        plural = _INFLECT.plural_noun(word)
        if plural and plural not in entries:
            entries[plural] = category

    ordered = sorted(
        entries.items(),
        key=lambda pair: (len(pair[0].split()), len(pair[0])),
        reverse=True,
    )
    return [
        (
            re.compile(
                r'(?<![A-Za-z])' + re.escape(form) + r'(?![A-Za-z])',
                re.IGNORECASE,
            ),
            category,
        )
        for form, category in ordered
    ]


def _object_mentions(
    caption: str,
    patterns: Sequence[tuple[re.Pattern, str]],
) -> list[tuple[str, str, int, int]]:
    """Find non-overlapping synonym spans as (matched_text, category, start, end)."""
    occupied: list[tuple[int, int]] = []
    mentions: list[tuple[str, str, int, int]] = []
    for pattern, category in patterns:
        for match in pattern.finditer(caption):
            start, end = match.span()
            if any(start < oe and os_ < end for os_, oe in occupied):
                continue
            occupied.append((start, end))
            mentions.append((match.group(0), category, start, end))
    return sorted(mentions, key=lambda m: m[2])


# ---------------------------------------------------------------------------
# Grammar-aware replacement
# ---------------------------------------------------------------------------
def _starts_with_vowel_sound(word: str) -> bool:
    """Rough heuristic for a/an."""
    return word[:1].lower() in 'aeiou'


def _apply_replacement(
    caption: str, start: int, end: int, source_text: str, target: str,
) -> str:
    """Replace source at [start:end] with target, preserving case and fixing a/an."""
    if source_text[:1].isupper():
        replacement = target[:1].upper() + target[1:]
    else:
        replacement = target

    prefix = caption[:start]
    art_match = re.search(r'\b(an?|An?)\s+$', prefix)
    if art_match:
        art_start = art_match.start(1)
        old_art = art_match.group(1)
        if _starts_with_vowel_sound(replacement):
            new_art = 'An' if old_art[0].isupper() else 'an'
        else:
            new_art = 'A' if old_art[0].isupper() else 'a'
        return (
            caption[:art_start] + new_art
            + caption[art_match.end(1):start] + replacement + caption[end:]
        )
    return caption[:start] + replacement + caption[end:]


# ---------------------------------------------------------------------------
# Swap candidate selection
# ---------------------------------------------------------------------------
def _candidate_categories(
    source: str,
    image_objects: set[str],
    categories: Sequence[str],
    matrix: np.ndarray | None,
    category_to_index: Mapping[str, int] | None,
    random_swap: bool,
    rng: random.Random,
    top_k: int = 10,
) -> list[str]:
    """Return replacement candidates.

    Structured: sorted by **descending** co-occurrence with *source* (high
    co-occurrence = plausible hallucination = hard negative), then a random
    pick from the top-k for variety.
    Random ablation: uniformly shuffled.
    """
    candidates = [
        category for category in categories
        if category != source and category not in image_objects
    ]
    if random_swap or matrix is None or category_to_index is None:
        rng.shuffle(candidates)
        return candidates
    if source not in category_to_index:
        rng.shuffle(candidates)
        return candidates

    source_row = matrix[category_to_index[source]]
    ranked = sorted(
        candidates,
        key=lambda c: (-source_row[category_to_index[c]], c),
    )
    pool = ranked[:min(top_k, len(ranked))]
    rng.shuffle(pool)
    return pool + ranked[len(pool):]


# ---------------------------------------------------------------------------
# Swap generator
# ---------------------------------------------------------------------------
def generate_swap(
    caption: str,
    image_objects: set[str],
    object_data: Mapping,
    patterns: Sequence[tuple[re.Pattern, str]],
    matrix: np.ndarray | None = None,
    category_to_index: Mapping[str, int] | None = None,
    *,
    random_swap: bool = False,
    rng: random.Random | None = None,
    top_k: int = 10,
) -> dict | None:
    """Generate one negative by replacing an object mention with an absent category.

    Uses precomputed *patterns* for span finding (including plural forms),
    fixes grammar (a/an, capitalisation), and validates with caption_to_objects
    that CHAIR will actually detect the injected target.  If the first
    candidate collides with surrounding text (e.g. inserting "bear" next to
    "teddy" creates "teddy bear"), the next candidate is tried automatically.
    """
    rng = rng or random.Random()
    mentions = _object_mentions(caption, patterns)
    if not mentions:
        return None

    # try mentions in random order so retries explore different positions
    mention_indices = list(range(len(mentions)))
    rng.shuffle(mention_indices)

    for mi in mention_indices:
        source_text, source, start, end = mentions[mi]
        candidates = _candidate_categories(
            source, image_objects, object_data['categories'],
            matrix, category_to_index, random_swap, rng, top_k,
        )
        for target in candidates:
            new_caption = _apply_replacement(caption, start, end, source_text, target)
            _, neg_objects = caption_to_objects(new_caption, object_data)
            if target in neg_objects:
                return {
                    'caption': new_caption,
                    'source': source,
                    'target': target,
                    'source_text': source_text,
                    'random_swap': random_swap,
                }
    return None


# ---------------------------------------------------------------------------
# Spot-check negatives
# ---------------------------------------------------------------------------
def spot_check_swaps(
    examples: Sequence[tuple[str, set[str]]],
    object_data: Mapping,
    patterns: Sequence[tuple[re.Pattern, str]],
    matrix: np.ndarray,
    category_to_index: Mapping[str, int],
    *,
    n: int = 100,
    seed: int = 0,
    random_swap: bool = False,
    verbose: int = 5,
) -> dict:
    """Generate *n* actual negatives and validate each with caption_to_objects.

    Keeps sampling until n negatives are produced (not n inputs).
    Prints the first *verbose* examples for manual inspection.
    """
    rng = random.Random(seed)
    pool = list(examples)
    rng.shuffle(pool)

    checked = 0
    tried = 0
    failures = []
    printed = 0

    for caption, image_objects in pool:
        if checked >= n:
            break
        tried += 1
        result = generate_swap(
            caption, image_objects, object_data, patterns,
            matrix, category_to_index,
            random_swap=random_swap, rng=rng,
        )
        if result is None:
            continue
        checked += 1

        # validate with CHAIR-compatible extraction
        _, neg_objects = caption_to_objects(result['caption'], object_data)
        target_found = result['target'] in neg_objects
        target_absent = result['target'] not in image_objects
        source_same = result['source'] == result['target']

        ok = target_found and target_absent and not source_same
        if not ok:
            failures.append({
                **result,
                'target_in_neg_caption': target_found,
                'target_in_image': not target_absent,
            })

        if printed < verbose:
            tag = 'OK' if ok else 'FAIL'
            print(f'  [{tag}] "{caption}"')
            print(f'       -> "{result["caption"]}"')
            print(f'       {result["source"]} -> {result["target"]}')
            printed += 1

    return {'requested': n, 'tried': tried, 'checked': checked, 'failures': failures}


# ---------------------------------------------------------------------------
# Loss functions
# ---------------------------------------------------------------------------
def caption_nll(model, tokenizer, prefix, caption: str, prefix_length: int = 40):
    """Teacher-forced GPT-2 NLL for one caption and one raw CLIP feature.

    No EOS is appended: ClipCap trains without one (pads with 0,
    ignore_index=0; generation stops at ".").
    """
    if prefix.ndim == 1:
        prefix = prefix.unsqueeze(0)
    prefix = prefix.float()
    prefix = prefix / prefix.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    token_ids = tokenizer.encode(caption, add_special_tokens=False)
    if not token_ids:
        raise ValueError('caption produced no tokens')
    tokens = torch.tensor(token_ids, device=prefix.device, dtype=torch.long).unsqueeze(0)
    prefix_embeds = model.clip_project(prefix).reshape(1, prefix_length, -1)
    token_embeds = model.gpt.transformer.wte(tokens)
    inputs_embeds = torch.cat((prefix_embeds, token_embeds), dim=1)
    labels = torch.cat(
        (
            torch.full((1, prefix_length), -100, device=tokens.device, dtype=torch.long),
            tokens,
        ),
        dim=1,
    )
    return model.gpt(inputs_embeds=inputs_embeds, labels=labels).loss


def object_swap_loss(
    model,
    tokenizer,
    prefixes,
    captions: Sequence[str],
    negative_captions: Sequence[str],
    *,
    prefix_length: int = 40,
    margin: float = 0.2,
    ranking_weight: float = 1.0,
):
    """Return positive NLL, negative NLL, ranking term, and total loss."""
    if len(captions) != len(negative_captions) or len(captions) != len(prefixes):
        raise ValueError('prefixes, captions, and negative_captions must have equal length')
    positive = torch.stack([
        caption_nll(model, tokenizer, prefixes[i], captions[i], prefix_length)
        for i in range(len(captions))
    ])
    negative = torch.stack([
        caption_nll(model, tokenizer, prefixes[i], negative_captions[i], prefix_length)
        for i in range(len(captions))
    ])
    ranking = F.relu(margin + positive - negative).mean()
    return {
        'positive_nll': positive.mean(),
        'negative_nll': negative.mean(),
        'ranking_loss': ranking,
        'loss': positive.mean() + ranking_weight * ranking,
    }


def smoke_test_loss(
    model, tokenizer, val_data, object_data, patterns,
    matrix, category_to_index,
) -> dict[str, float]:
    """Smoke-test the loss with real val captions paired to their own features.

    Finds two captions with swappable objects, generates negatives, and verifies
    the loss is finite and positive NLL is lower than negative NLL.
    """
    device = next(model.parameters()).device
    rng = random.Random(0)

    captions_used = []
    negatives_used = []
    prefix_rows = []

    for cap_entry in val_data['captions']:
        if len(captions_used) >= 2:
            break
        caption = cap_entry['caption']
        _, objects = caption_to_objects(caption, object_data)
        if not objects:
            continue
        result = generate_swap(
            caption, set(), object_data, patterns,
            matrix, category_to_index, rng=rng,
        )
        if result is None:
            continue
        captions_used.append(caption)
        negatives_used.append(result['caption'])
        prefix_rows.append(cap_entry['clip_embedding'])

    assert len(captions_used) == 2, f'need 2 swappable captions, got {len(captions_used)}'
    prefixes = val_data['clip_embedding'][prefix_rows].to(device)
    values = object_swap_loss(model, tokenizer, prefixes, captions_used, negatives_used)
    out = {name: float(value.detach().cpu()) for name, value in values.items()}

    for i in range(2):
        print(f'  [{i}] pos: "{captions_used[i]}"')
        print(f'       neg: "{negatives_used[i]}"')
    print(f'  positive_nll={out["positive_nll"]:.3f}  negative_nll={out["negative_nll"]:.3f}')
    print(f'  ranking_loss={out["ranking_loss"]:.3f}  total_loss={out["loss"]:.3f}')

    return out

## Object list and co-occurrence matrix

In [9]:
object_data = load_object_data(os.path.join(TRACK1_ROOT, 'eval', 'coco_objects.json'))
chair_gt = load_chair_gt(os.path.join(TRACK1_ROOT, 'eval', 'chair_gt_train.pkl'))
categories = object_data['categories']
cooccurrence, category_to_index = build_cooccurrence_matrix(chair_gt, categories)

matrix_path = os.path.join(TRACK3_ROOT, 'eval/cooccurrence_train.pkl')
save_cooccurrence_matrix(matrix_path, cooccurrence, categories)
print('categories:', len(categories))
print('images:', len(chair_gt))
print('matrix:', cooccurrence.shape, cooccurrence.dtype)
print('saved:', matrix_path)
assert np.array_equal(cooccurrence, cooccurrence.T)
# each object co-occurs with itself at least as often as with any other
diag = np.diag(cooccurrence)
off_diag_max = (cooccurrence - np.diag(diag)).max(axis=1)
assert np.all(diag >= off_diag_max), 'diagonal must be >= every off-diagonal in its row'

# precompute regex patterns for span-based object finding (includes plural forms)
patterns = build_replacement_patterns(object_data)
print('replacement patterns:', len(patterns))

categories: 80
images: 113287
matrix: (80, 80) int64
saved: /content/drive/MyDrive/sc4001_nndl_grp_project/phase_1/track_3/eval/cooccurrence_train.pkl
replacement patterns: 796


The truth for each image is `seg` union every object list in `cap`; therefore the diagonal counts images containing each object and off-diagonal entries count image-level co-occurrence.

## Swap generation and 100-negative spot checks

In [10]:
with open(os.path.join(TRACK1_ROOT, 'data', 'karpathy_index.json')) as handle:
    karpathy_index = json.load(handle)

examples = []
for row in karpathy_index:
    if row['split'] != 'train' or row['image_id'] not in chair_gt:
        continue
    image_objects = true_objects(chair_gt[row['image_id']])
    for caption in row['captions']:
        examples.append((caption, image_objects))

print('structured swaps:')
structured_check = spot_check_swaps(
    examples, object_data, patterns, cooccurrence, category_to_index,
    n=100, seed=0, verbose=5,
)
print()
print('random swaps:')
random_check = spot_check_swaps(
    examples, object_data, patterns, cooccurrence, category_to_index,
    n=100, seed=0, random_swap=True, verbose=5,
)
print()
print('structured:', {k: v for k, v in structured_check.items() if k != 'failures'})
print('random:', {k: v for k, v in random_check.items() if k != 'failures'})
if structured_check['failures']:
    print('STRUCTURED FAILURES:', structured_check['failures'])
if random_check['failures']:
    print('RANDOM FAILURES:', random_check['failures'])
assert not structured_check['failures'], 'structured swap validation failed'
assert not random_check['failures'], 'random swap validation failed'
assert structured_check['checked'] == 100, f'only got {structured_check["checked"]} structured swaps'
assert random_check['checked'] == 100, f'only got {random_check["checked"]} random swaps'

structured swaps:
  [OK] "A large passenger jet sitting on top of a rural runway."
       -> "A large handbag jet sitting on top of a rural runway."
       person -> handbag
  [OK] "A cruise ship docked for letting passengers off to port"
       -> "A cruise car docked for letting passengers off to port"
       boat -> car
  [OK] "Two zebras are sharing grass from a pile."
       -> "Two person are sharing grass from a pile."
       zebra -> person
  [OK] "A black and white cat inside a brick building looking out."
       -> "A black and white couch inside a brick building looking out."
       cat -> couch
  [OK] "a girl in a pink top is training a horse"
       -> "a girl in a pink top is training a cow"
       horse -> cow

random swaps:
  [OK] "A large passenger jet sitting on top of a rural runway."
       -> "A large cell phone jet sitting on top of a rural runway."
       person -> cell phone
  [OK] "A cruise ship docked for letting passengers off to port"
       -> "A cruise shi

In [11]:
example_caption, example_objects = examples[0]
demo_rng = random.Random(0)
structured = generate_swap(
    example_caption, example_objects, object_data, patterns,
    cooccurrence, category_to_index, rng=demo_rng,
)
random_negative = generate_swap(
    example_caption, example_objects, object_data, patterns,
    cooccurrence, category_to_index, random_swap=True, rng=demo_rng,
)
print('original:', example_caption)
print('structured:', structured)
print('random:', random_negative)

original: Closeup of bins of food that include broccoli and bread.
structured: {'caption': 'Closeup of bins of food that include bottle and bread.', 'source': 'broccoli', 'target': 'bottle', 'source_text': 'broccoli', 'random_swap': False}
random: {'caption': 'Closeup of bins of food that include zebra and bread.', 'source': 'broccoli', 'target': 'zebra', 'source_text': 'broccoli', 'random_swap': True}


## Pretrained ClipCap loss smoke test

Uses real val captions paired to their own CLIP features.  `caption_nll` performs the required L2 normalization internally.

In [12]:
clipcap_dir = '/content/CLIP_prefix_caption'

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required for the pretrained ClipCap smoke test.')
device = 'cuda'

checkpoint_path = os.path.join(
    TRACK1_ROOT, 'models', 'clipcap_transformer_coco_pretrained.pt'
)
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        f'Pretrained checkpoint not found at {checkpoint_path}. Run Track 1 checkpoint setup first.'
    )

PREFIX_LENGTH = 40
ckpt_model = ClipCaptionPrefix(
    PREFIX_LENGTH,
    clip_length=40,
    prefix_size=640,
    num_layers=8,
    mapping_type=MappingType.Transformer,
)
state = torch.load(checkpoint_path, map_location='cpu')
load_result = ckpt_model.load_state_dict(state, strict=False)
assert not load_result.missing_keys, load_result.missing_keys
for key in load_result.unexpected_keys:
    assert key.endswith('.attn.bias') or key.endswith('.attn.masked_bias'), key
ckpt_model = ckpt_model.to(device).eval()
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

feature_path = os.path.join(TRACK1_ROOT, 'features', 'coco_rn50x4_val.pkl')
with open(feature_path, 'rb') as handle:
    val_data = pickle.load(handle)

loss_values = smoke_test_loss(
    ckpt_model, tokenizer, val_data, object_data, patterns,
    cooccurrence, category_to_index,
)
print(loss_values)
assert all(np.isfinite(v) for v in loss_values.values())
assert loss_values['positive_nll'] < loss_values['negative_nll'], \
    'positive NLL should be lower than negative NLL for real caption pairs'

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  [0] pos: "A woman stands in the dining area at the table."
       neg: "A bench stands in the dining area at the table."
  [1] pos: "A room with chairs, a table, and a woman in it."
       neg: "A room with chairs, a knife, and a woman in it."
  positive_nll=2.053  negative_nll=3.183
  ranking_loss=0.000  total_loss=2.053
{'positive_nll': 2.0532782077789307, 'negative_nll': 3.1833786964416504, 'ranking_loss': 0.0, 'loss': 2.0532782077789307}
